# Common Crawl's URL Index via Hugging Face

[Common Crawl's URL Index](https://commoncrawl.org/url-index) (previously known as Columnar Index) is one of the indexes available for querying the Common Crawl corpus. As the name suggests, it is an index to the WARC files and URLs in the Common Crawl corpus in a columnar format (Apache Parquet). This format of the index is suited to efficient analytical and/or bulk queries of the data, saving time and computing resources.

The index files are also available on the Common Crawl HF bucket: https://huggingface.co/buckets/commoncrawl/commoncrawl

The [duckdb](https://duckdb.org/) can be used to query the index without downloading all the data.

## Setup

Follow the [official instructions](https://duckdb.org/install/) to install the duckdb CLI:

```bash
curl https://install.duckdb.org | sh
```

## Schema

Let's first look at the schema of the index:


In [1]:
# via CLI
!duckdb -c "DESCRIBE FROM read_parquet('https://huggingface.co/buckets/commoncrawl/commoncrawl/resolve/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-17/subset=warc/part-00000-ee7620e8-27a7-4c74-846b-a92b039dae92.c000.gz.parquet')"

]11;?

┌─────────────────────────────────────────────────────┐
│                      Describe                       │
│                                                     │
│ url_surtkey                varchar                  │
│ url                        varchar                  │
│ url_host_name              varchar                  │
│ url_host_tld               varchar                  │
│ url_host_2nd_last_part     varchar                  │
│ url_host_3rd_last_part     varchar                  │
│ url_host_4th_last_part     varchar                  │
│ url_host_5th_last_part     varchar                  │
│ url_host_registry_suffix   varchar                  │
│ url_host_registered_domain varchar                  │
│ url_host_private_suffix    varchar                  │
│ url_host_private_domain    varchar                  │
│ url_host_name_reversed     varchar                  │
│ url_protocol               varchar                  │
│ url_port                   integer            

In [2]:
# via Python client
import duckdb

index_file = "https://huggingface.co/buckets/commoncrawl/commoncrawl/resolve/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-17/subset=warc/part-00000-ee7620e8-27a7-4c74-846b-a92b039dae92.c000.gz.parquet"

duckdb.sql('INSTALL httpfs; LOAD httpfs;')
duckdb.sql(
    f"DESCRIBE FROM read_parquet('{index_file}')"
).show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ url_surtkey                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url                        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_host_name              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_host_tld               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_host_2nd_last_part     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_host_3rd_last_part     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_host_4th_last_part     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_host_5th_last_part     │ VARCHAR     │ YES     │ NULL    │ 

In [3]:
# The index uses hive-partitioning for `crawl` and `subset` that are automatically constructed if you would use the S3 index.
# However, we are using HF via HTTP so we need to manually construct the partitions using glob from the HF API.

from huggingface_hub import hffs

# NOTE: We are taking on a sample of all files (part-0000-*)
index_files = list(hffs.glob("buckets/commoncrawl/commoncrawl/cc-index/table/cc-main/warc/crawl=*/subset=*/part-00000-*.parquet"))

print(f"Found {len(index_files):,} index files on HF bucket")

Found 9 index files on HF bucket


In [4]:
# Rewrite bucket paths to HTTP URLs
http_index_files = [f.replace("buckets/commoncrawl/commoncrawl/", "https://huggingface.co/buckets/commoncrawl/commoncrawl/resolve/") for f in index_files]

http_index_files

['https://huggingface.co/buckets/commoncrawl/commoncrawl/resolve/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-17/subset=crawldiagnostics/part-00000-ee7620e8-27a7-4c74-846b-a92b039dae92.c000.gz.parquet',
 'https://huggingface.co/buckets/commoncrawl/commoncrawl/resolve/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-17/subset=robotstxt/part-00000-ee7620e8-27a7-4c74-846b-a92b039dae92.c000.gz.parquet',
 'https://huggingface.co/buckets/commoncrawl/commoncrawl/resolve/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-17/subset=warc/part-00000-ee7620e8-27a7-4c74-846b-a92b039dae92.c000.gz.parquet',
 'https://huggingface.co/buckets/commoncrawl/commoncrawl/resolve/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-21/subset=crawldiagnostics/part-00000-0ddabc35-a36d-4326-aa24-6ee6775acdc0.c000.gz.parquet',
 'https://huggingface.co/buckets/commoncrawl/commoncrawl/resolve/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-21/subset=robotstxt/part-00000-0ddabc35-a36d-4326-aa24-6ee6775acdc0.c000.gz.parquet',
 

In [5]:
# Let's see how large our sample is
duckdb.sql(
    f"SELECT crawl, subset, COUNT(*) FROM read_parquet({http_index_files!r}, hive_partitioning = true) GROUP BY crawl, subset"
).show()

┌─────────────────┬──────────────────┬──────────────┐
│      crawl      │      subset      │ count_star() │
│     varchar     │     varchar      │    int64     │
├─────────────────┼──────────────────┼──────────────┤
│ CC-MAIN-2026-25 │ robotstxt        │       145762 │
│ CC-MAIN-2026-25 │ warc             │      6890137 │
│ CC-MAIN-2026-21 │ warc             │      7081731 │
│ CC-MAIN-2026-25 │ crawldiagnostics │      1491237 │
│ CC-MAIN-2026-21 │ crawldiagnostics │      1554206 │
│ CC-MAIN-2026-21 │ robotstxt        │       158868 │
│ CC-MAIN-2026-17 │ robotstxt        │       266481 │
│ CC-MAIN-2026-17 │ warc             │      7165967 │
│ CC-MAIN-2026-17 │ crawldiagnostics │      1534027 │
└─────────────────┴──────────────────┴──────────────┘



## Examples

### Query 1: Counting Pages Per Domain

Time to run the first query! This query counts the number of pages per domain within a single top-level domain, `.ru` (Russia):



In [6]:
duckdb.sql(f"""SELECT COUNT(*) AS count,
       url_host_registered_domain
FROM read_parquet({http_index_files!r}, hive_partitioning = true) 
WHERE crawl = 'CC-MAIN-2026-17' AND subset = 'warc'
  AND url_host_tld = 'ru'
GROUP BY  url_host_registered_domain
HAVING (COUNT(*) >= 100)
ORDER BY  count DESC""").show()

┌────────┬────────────────────────────┐
│ count  │ url_host_registered_domain │
│ int64  │          varchar           │
├────────┼────────────────────────────┤
│ 166381 │ yandex.ru                  │
│  28067 │ zr.ru                      │
│  23968 │ zin.ru                     │
│  18272 │ zarplata.ru                │
│  16030 │ zab.ru                     │
│  13283 │ yugzone.ru                 │
│  12003 │ yapl.ru                    │
│  10861 │ zp-news.ru                 │
│  10348 │ zavod-metakon.ru           │
│   9794 │ zhilibyli.ru               │
│     ·  │      ·                     │
│     ·  │      ·                     │
│     ·  │      ·                     │
│    100 │ zamena-masla-oilcity.ru    │
│    100 │ zov-vrn.ru                 │
│    100 │ zoomag174.ru               │
│    100 │ yushkovpartners.ru         │
│    100 │ zharim-myaso.ru            │
│    100 │ zvuk4event.ru              │
│    100 │ zdraviepro.ru              │
│    100 │ yotad.ru                   │


### Query 2: Finding Domains with Multilingual Content

The next example demonstrates the power of DuckDB functions by looking for domains with multilingual content. It uses regular expressions to identify [ISO 639-1](https://www.iso.org/iso-639-language-code) language codes in the URL: e.g. the "en" in `https://example.com/about/en/page.html`.

For demonstration purposes, the query is restricted to a single top-level domain (`.ru`). Run the following query:



In [7]:
duckdb.query(f"""SELECT url_host_registered_domain AS domain,
       COUNT(DISTINCT(url_path_lang)) as n_lang,
       COUNT(*) as n_pages,
       histogram(url_path_lang) as lang_counts
FROM read_parquet({http_index_files!r}, hive_partitioning = true),
  UNNEST(regexp_extract_all(url_path, '/([a-z][a-z])/', 1)) AS t (url_path_lang)
WHERE crawl = 'CC-MAIN-2026-17'
  AND subset = 'warc'
  AND url_host_tld = 'ru'
GROUP BY url_host_registered_domain
HAVING COUNT(*) >= 100
   AND COUNT(DISTINCT(url_path_lang)) >= 2
ORDER BY n_pages DESC""").show()

┌────────────────────────┬────────┬─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│         domain         │ n_lang │ n_pages │                                                                                                                                                                                                            lang_counts                                                                                                                                                                                                            │
│        varchar         │ int64  │  int64  │                       

The result should contain four columns:

- domain: (e.g. yandex.ru)
- n_lang: (a count of distinct languages detected in each domain)
- n_pages: (a count of matched language-code path elements in each domain; a page with several such elements is counted more than once)
- lang_counts: (a hash map of language codes and their counts, e.g. {de=2698, hi=1, ru=13, ...})

The magic is done by the UNNEST command, which first extracts all two-letter path elements (e.g., `/en/`), then unrolls the elements into a new column `url_path_lang`. If two or more path elements are found, it creates multiple rows. Finally, it counts pages and unique languages and generates a histogram of language codes.

## See also

Further information can be found at the documentation of the URL index:

https://commoncrawl.org/url-index
